# TRIBE v2 — Puntaje de activación cerebral por video (Colab)

Prueba el modelo **TRIBE v2** (V‑JEPA2 + LLaMA‑3.2 + Wav2Vec2‑BERT) sin servidor ni Docker: subes un video y obtienes una curva de *activación cerebral* **A(t)**.

**Antes de empezar:**
1. **Runtime con GPU**: menú `Entorno de ejecución → Cambiar tipo de entorno → GPU`. Necesitas ≥16 GB de VRAM (la T4 gratuita va justa; L4/A100 de Colab Pro van holgadas).
2. **Token de Hugging Face**: LLaMA‑3.2 es *gated*. Crea un token en https://huggingface.co/settings/tokens y **acepta la licencia** del modelo (p. ej. https://huggingface.co/meta-llama/Llama-3.2-3B).
3. Usa **clips cortos** (pocos segundos) en las primeras pruebas: VRAM y tiempo crecen con la duración.

> **Nota honesta sobre la resolución:** la señal BOLD es lenta; el puntaje *real* es uno por **TR (1.49 s)**. Las opciones `second / frame / ms` son **interpolación** de esa curva, no resolución nueva.
>
> **Licencia:** los pesos de TRIBE v2 son **CC BY‑NC** (uso no comercial).

## 1) Verificar la GPU

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "|", round(p.total_memory / 1e9, 1), "GB VRAM")
else:
    print("\u26a0\ufe0f No hay GPU. Ve a 'Entorno de ejecución → Cambiar tipo de entorno → GPU'.")

## 2) Instalar TRIBE v2 y dependencias

Se instala `tribev2` junto con `torch / torchvision / torchaudio` **alineados a 2.6.0**. Esa alineación evita el error `undefined symbol: aoti_torch_abi_version`, que aparece cuando Colab deja un `torchaudio` más nuevo que el `torch` que pide tribev2. Además `tribev2` fija `numpy==2.2.6`.

Como cambian `torch` y `numpy` (ya cargados en memoria), al terminar esta celda **reinicia el entorno** (`Entorno de ejecución → Reiniciar sesión`, o el botón **RESTART RUNTIME**) y continúa desde el paso 3 — **no repitas esta celda**.

In [ ]:
%pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 "git+https://github.com/facebookresearch/tribev2" scipy matplotlib pandas
print("\nListo (torch/torchaudio/torchvision alineados a 2.6.0). ⚠️ Si Colab muestra 'RESTART RUNTIME', púlsalo y sigue desde el paso 3.")

## 3) Login en Hugging Face (LLaMA es *gated*)

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Pega tu HF_TOKEN (no se mostrará): "))
print("Login OK")

## 4) Cargar el modelo

La primera vez descarga decenas de GB de pesos a `./cache`; puede tardar varios minutos.

In [ ]:
from tribev2 import TribeModel
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Modelo cargado.")

## 5) Funciones del puntaje

Mismo cálculo que la API. Por defecto **RMS espacial** A(t) = √(media_v P[t,v]²), cuyo valor basal es ≈ 1 (cada vértice está z‑scoreado).

In [ ]:
import numpy as np
from scipy.interpolate import interp1d


def compute_activation_curve(preds, method="rms", theta=1.96, roi_mask=None):
    """A(t) a partir de preds (T, V). method: 'rms' | 'abs' | 'thr'."""
    P = np.asarray(preds, dtype=np.float64)
    if P.ndim != 2:
        raise ValueError(f"Se esperaba preds 2D (T, V), llegó {P.shape}")
    if roi_mask is not None:
        P = P[:, roi_mask]
    if method == "rms":
        return np.sqrt(np.mean(P ** 2, axis=1))          # basal ~1
    if method == "abs":
        return np.mean(np.abs(P), axis=1)                # basal ~0.798
    if method == "thr":
        return np.mean((np.abs(P) >= theta), axis=1)     # fracción "activa"
    raise ValueError(f"método desconocido: {method}")


def normalize_curve(A, mode="none"):
    if mode == "none":
        return A
    if mode == "zscore":
        mu, sd = A.mean(), A.std()
        return (A - mu) / sd if sd > 0 else A - mu
    if mode == "minmax":
        lo, hi = A.min(), A.max()
        return (A - lo) / (hi - lo) if hi > lo else np.zeros_like(A)
    raise ValueError(f"normalización desconocida: {mode}")


def extract_timestamps(segments, n_rows, tr=1.49):
    """Tiempo (s) de cada fila. predict() devuelve una LISTA de objetos-segmento
    (.start/.offset/.duration). Punto medio real = start + offset + duration/2.
    Si tu versión ya incluye offset en .start, quita ese sumando (timestamps
    que avancen ~2*TR en vez de ~TR = doble conteo)."""
    try:
        ts = []
        for s in segments:
            start = getattr(s, "start", None)
            if start is None:
                raise AttributeError
            off = float(getattr(s, "offset", 0.0) or 0.0)
            dur = float(getattr(s, "duration", tr) or tr)
            ts.append(float(start) + off + dur / 2.0)
        if len(ts) == n_rows:
            return np.asarray(ts)
    except (TypeError, AttributeError, ValueError):
        pass
    try:
        seg = np.asarray(segments, dtype=np.float64)
        if seg.ndim == 2 and seg.shape[0] == n_rows and seg.shape[1] >= 2:
            return (seg[:, 0] + seg[:, 1]) / 2.0
        if seg.ndim == 1 and seg.shape[0] == n_rows:
            return seg
    except (TypeError, ValueError):
        pass
    return np.arange(n_rows, dtype=np.float64) * tr


def resample_curve(t_native, A_native, granularity="tr", fps=None):
    """Interpola A(t) a la rejilla pedida. Devuelve (t, A, interpolado?)."""
    if granularity == "tr":
        return t_native, A_native, False
    t0, t1 = float(t_native[0]), float(t_native[-1])
    if granularity == "second":
        grid = np.arange(np.floor(t0), np.floor(t1) + 1, 1.0)
    elif granularity == "frame":
        if not fps or fps <= 0:
            raise ValueError("granularity='frame' requiere fps > 0")
        grid = np.arange(int(t0 * fps), int(t1 * fps) + 1) / fps
    elif granularity == "ms":
        grid = np.arange(int(t0 * 1000), int(t1 * 1000) + 1) / 1000.0
    else:
        raise ValueError(f"granularidad desconocida: {granularity}")
    f = interp1d(t_native, A_native, kind="linear", bounds_error=False,
                 fill_value=(A_native[0], A_native[-1]))
    return grid, f(grid), True

print("Funciones listas.")

## 6) Subir un video y calcular el puntaje

In [ ]:
from google.colab import files
up = files.upload()                  # elige un video corto (con audio)
video_path = next(iter(up))
print("Video:", video_path)

In [ ]:
df = model.get_events_dataframe(video_path=video_path)
preds, segments = model.predict(events=df)
preds = np.asarray(preds)
print("preds (T, V):", preds.shape)

# Verifica UNA vez la convención de tiempos (ver docstring de extract_timestamps):
try:
    print("segmento[0]:", vars(segments[0]))
except TypeError:
    print("segmento[0]:", segments[0])

# --- parámetros ---
method        = "rms"      # "rms" | "abs" | "thr"
normalization = "none"     # "none" | "zscore" | "minmax"
granularity   = "tr"       # "tr" | "second" | "frame" | "ms"
fps           = None       # requerido si granularity="frame"

A = normalize_curve(compute_activation_curve(preds, method=method), normalization)
t = extract_timestamps(segments, len(A))
t_grid, A_grid, interpolated = resample_curve(t, A, granularity, fps)
print(f"{len(A_grid)} puntos | interpolado={interpolated}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(11, 4))
plt.plot(t_grid, A_grid, lw=1.5)
if method == "rms" and normalization == "none":
    plt.axhline(1.0, ls="--", c="gray", lw=0.8, label="basal RMS \u2248 1")
    plt.legend()
plt.xlabel("tiempo (s)"); plt.ylabel(f"activación ({method})")
plt.title("Puntaje de activación cerebral por instante")
plt.grid(alpha=0.3); plt.show()

In [ ]:
import pandas as pd
out = pd.DataFrame({"t_seconds": np.round(t_grid, 4), "score": np.round(A_grid, 6)})
out.to_csv("activation.csv", index=False)
print("Guardado activation.csv")
# files.download("activation.csv")   # descomenta para descargar
out.head()

## Solución de problemas

- **`undefined symbol: aoti_torch_abi_version` (al cargar el modelo)**: desajuste `torch`/`torchaudio`. Reinstala la terna alineada `%pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0`, **reinicia el entorno** y reanuda desde el paso 3.
- **`CUDA out of memory`**: usa un clip más corto o un runtime con más VRAM (Colab Pro: L4 24 GB / A100 40 GB).
- **Error con `bfloat16` en una T4**: la T4 (Turing) no soporta bf16; usa un runtime **L4/A100**.
- **`401` / *gated* al cargar**: acepta la licencia del modelo en su página de Hugging Face con la **misma cuenta** del token.
- **Recuerda**: `second/frame/ms` son interpolación; el dato real es 1 punto por **TR = 1.49 s**.